In [1]:
import xml.etree.ElementTree as ET
import os, tempfile, shutil

from pycparser.ply.yacc import error_count

In [2]:

class TOCmanager:
    def __init__(self, file='all_journals_toc.xml'):
        self.file = file
        self.tree = ET.parse(file)
        self.root = self.tree.getroot()
        self.journal2index = {j.attrib['name']: i for i, j in enumerate(self.root)}
        self.journalsList = list(self.journal2index.keys())
        self.journal_tot = [len(j) for j in self.root]
        self.tot = sum(self.journal_tot)
        self.reverse_gaids = dict()
        self.gaids = dict()
        gaid = 0
        for jid, tot in enumerate(self.journal_tot):
            for aid in range(tot):
                self.gaids[gaid] = (jid, aid)
                self.reverse_gaids[(jid, aid)] = gaid
                gaid += 1

    def get(self, jid=0, aid=0):
        if aid >= self.journal_tot[jid]:
            print(f'Journal {self.journalsList[jid]} (JID:{jid}) has {self.journal_tot[jid]} articles. AID requested: {aid}')
            return None
        article = {elem.tag: elem.text for elem in self.root[jid][aid]}
        if len(article.get('Abstract', '') or '') < 100:
            article['Abstract'] = ''
        article['Journal'] = self.journalsList[jid]
        article['jid'] = jid
        article['aid'] = aid
        article['gaid'] = self.reverse_gaids[(jid, aid)]
        return article

    def gaid(self, gaid):
        jid, aid = self.gaids[gaid]
        return self.get(jid, aid)

    def gaid_batch(self, gaids):
        return [self.get(*self.gaids[gaid]) for gaid in gaids]

    def gabj(self,jid):
        return [{article.tag:article.text for article in elem} for elem in tocs.root[jid]]

    def add_field(self, jid, aid, field_name, value):
        """Add or update a field (e.g. keywords) in a specific article."""
        article_elem = self.root[jid][aid]
        existing = article_elem.find(field_name)
        if existing is not None:
            existing.text = value
        else:
            new_elem = ET.SubElement(article_elem, field_name)
            new_elem.text = value

    def save(self, path=None):
        """Safely write XML to file. Abort if any element text is not str or None."""
        target = path or self.file
        # verify all text fields
        for elem in self.root.iter():
            if elem.text is not None and not isinstance(elem.text, str):
                raise TypeError(f"Invalid text type in <{elem.tag}>: {type(elem.text).__name__}")

        # safe write to temporary file
        tmp_fd, tmp_path = tempfile.mkstemp(suffix=".xml", prefix="tmp_", dir=os.path.dirname(target) or ".")
        os.close(tmp_fd)
        try:
            self.tree.write(tmp_path, encoding='utf-8', xml_declaration=True)
            shutil.move(tmp_path, target)
        except Exception:
            if os.path.exists(tmp_path):
                os.remove(tmp_path)
            raise

    def print(self, article):
        print(f"### Article - JID: {article['jid']} AID: {article['aid']} - GAID: {article['gaid']} ###")
        for k, t in article.items():
            if k in ('aid', 'jid', 'gaid'):
                continue
            print(f"\t{k} : {t}")

    def info(self):
        print("### TOCS ###")
        print(f"\tJournals: {len(self.journal_tot)}")
        print(f"\tArticles: {self.tot}")

    def __str__(self):
        self.info()
        return ''

    def add_field_gaid(self, gaid, field_name, value):
        """Add or update a field in an article given its global article ID (gaid)."""
        if gaid not in self.gaids:
            print(f"Invalid GAID: {gaid}")
            return
        jid, aid = self.gaids[gaid]
        self.add_field(jid, aid, field_name, value)

    def delete_article(self, jid, aid):
        """Delete one article by journal and article index."""
        if jid >= len(self.root):
            print(f"Invalid journal ID: {jid}")
            return
        if aid >= len(self.root[jid]):
            print(f"Journal {self.journalsList[jid]} has only {len(self.root[jid])} articles.")
            return
        # Remove the element
        del self.root[jid][aid]
        # Update bookkeeping
        self.journal_tot[jid] = len(self.root[jid])
        self.tot = sum(self.journal_tot)
        # Rebuild GAID mappings
        self.gaids.clear()
        self.reverse_gaids.clear()
        gaid = 0
        for j, tot in enumerate(self.journal_tot):
            for a in range(tot):
                self.gaids[gaid] = (j, a)
                self.reverse_gaids[(j, a)] = gaid
                gaid += 1
        print(f"Deleted article {aid} from journal {self.journalsList[jid]}")

    def delete_article_gaid(self, gaid):
        """Delete one article using its global article ID (GAID)."""
        if gaid not in self.gaids:
            print(f"Invalid GAID: {gaid}")
            return
        jid, aid = self.gaids[gaid]
        self.delete_article(jid, aid)

    def find_by_doi(self, doi):
        """Return (jid, aid) of an article with the given DOI."""
        for jid, journal in enumerate(self.root):
            for aid, article in enumerate(journal):
                doi_elem = article.find('DOI')
                if doi_elem is not None and doi_elem.text == doi:
                    return jid, aid
        return None

    def delete_by_doi(self, doi):
        """Delete an article by its DOI."""
        res = self.find_by_doi(doi)
        if not res:
            print(f"DOI not found: {doi}")
            return
        jid, aid = res
        self.delete_article(jid, aid)
        print(f"Deleted article with DOI {doi}")

    def add_field_doi(self, doi, field_name, value):
        """Add or update a field in an article identified by its DOI."""
        res = self.find_by_doi(doi)
        if not res:
            print(f"DOI not found: {doi}")
            return
        jid, aid = res
        self.add_field(jid, aid, field_name, value)



#import random
generalist = ['Cell', 'Nature','Science','Science Advances','eLife','Current Biology','Cell Reports','Cellular and Molecular Life Sciences','Communications Biology','International Journal of Molecular Sciences','iScience','Journal of Physiology','Scientific Reports']
tocs = TOCmanager()
tocs.info()
#article = tocs.gaid(10)
#tocs.print(article)
#gaids = random.sample(range(tocs.tot), 20)
#articles = tocs.gaid_batch(gaids)
#prepared = [(article['Title'],article['Abstract']) for article in articles]
#prepared

### TOCS ###
	Journals: 47
	Articles: 2259


In [3]:
import requests
import os
from falcon_tests import ArticleClassifierOllama, ArticleClassifier2

from dotenv import load_dotenv
load_dotenv()
host = os.getenv('HOST')
ollama = ArticleClassifier2()#ArticleClassifierOllama(host=host)
error_counter = 0

for journal,jid in tocs.journal2index.items():
    if jid==0:
        print('skipping biorxiv')
        continue
    print(f"##### {journal} - Articles: {tocs.journal_tot[jid]} #####")
    articles = tocs.gabj(jid)
    for aid,article in enumerate(articles):
        try:
            result = ollama.classify((article['Title'],article['Abstract']))
            if result['neuroscience']=='yes':
                tocs.add_field_doi(article['DOI'],'generic_keywords', ','.join(result['generic_keywords']))
                tocs.add_field_doi(article['DOI'],'specific_keywords', ','.join(result['specific_keywords']))
            else:
                tocs.delete_by_doi(article['DOI'])
            print(f"{jid}:{aid} - {article['Title']}")
            print(result)
            print('----')
        except Exception:
            error_counter +=1
            print(f"------> Ollama error!! {article.get('DOI','DOI not found')}<-------")

#if error_counter==0:
#    tocs.save()
#else:
#    print('Check errors!!!')
#tocs.save()

skipping biorxiv
##### Cell - Articles: 32 #####
1:0 - Structural insights into brain thyroid hormone transport via MCT8 and OATP1C1
{'neuroscience': 'yes', 'type': 'article', 'generic_keywords': ['thyroid hormone transport', 'brain', 'MCT8', 'OATP1C1'], 'specific_keywords': ['neurobiology of thyroid hormone', 'brain-specific transporters', 'molecular mechanisms']}
----
1:1 - A mast cell receptor mediates post-stroke brain inflammation via a dural-brain axis
{'neuroscience': 'yes', 'type': 'article', 'generic_keywords': ['neuroinflammation', 'neurovascular unit', 'stroke recovery'], 'specific_keywords': ['brain inflammation', 'dural-brain axis', 'post-stroke mechanisms']}
----
1:2 - Prevalent mesenchymal drift in aging and disease is reversed by partial reprogramming
{'neuroscience': 'yes', 'type': 'article', 'generic_keywords': ['aging and neurodegeneration', 'cell death mechanisms', 'developmental disorders', 'molecular neuroscience', 'stem cells'], 'specific_keywords': ['Mesenchymal

KeyboardInterrupt: 

In [10]:
 ollama.classify([(article['Title'],article['Abstract'])])

KeyboardInterrupt: 

In [27]:
articles = tocs.gabj(1)

In [42]:
kw = [article['generic_keywords'].split(',')+article['specific_keywords'].split(',') for article in articles]
kws = [k for keys in kw for k in keys]
from collections import Counter
Counter(kws).most_common(30)

[('neurodegeneration', 4),
 ('neurotransmission', 2),
 ('mast cells', 2),
 ('cytokines', 2),
 ('histamine', 2),
 ("Alzheimer's disease", 2),
 ('combination therapy', 2),
 ('microglia', 2),
 ('dopamine', 2),
 ('eukaryotic cells', 2),
 ('gene expression', 2),
 ('neuroscience', 2),
 ('immune system', 2),
 ('CD8+ T cells', 2),
 ('multiple sclerosis', 2),
 ('mitochondria', 2),
 ('inflammation', 2),
 ('brain', 1),
 ('thyroid hormone', 1),
 ('transport', 1),
 ('MCT8', 1),
 ('OATP1C1', 1),
 ('lipid metabolism', 1),
 ('brain structure', 1),
 ('hormonal regulation', 1),
 ('stroke', 1),
 ('brain inflammation', 1),
 ('dural-brain axis', 1),
 ('neuroinflammation', 1),
 ('perivascular spaces', 1)]